# OSL-Words Original Dataset Analysis
Analyzes the **original (non-augmented)** OSL-Words dataset before any splitting.
Sources:
- Original IDs are extracted from the augmented train label file (augmented names embed the original ID at the end)
- Dev and Test label files contain only original videos

Outputs saved to `../OSL_Run_Pipeline/reports/`

In [1]:
import gzip, pickle, re, json, os
import pandas as pd
from collections import defaultdict
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
UNISIGN_DATA = Path(r"C:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main\data\OSL-Words")
REPORT_DIR   = Path(r"C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def load_labels(path):
    with gzip.open(path, 'rb') as f:
        return pickle.load(f)

train_labels = load_labels(UNISIGN_DATA / 'labels-osl.train')
dev_labels   = load_labels(UNISIGN_DATA / 'labels-osl.dev')
test_labels  = load_labels(UNISIGN_DATA / 'labels-osl.test')

print(f"Train entries (augmented): {len(train_labels)}")
print(f"Dev entries  (original)  : {len(dev_labels)}")
print(f"Test entries (original)  : {len(test_labels)}")

Train entries (augmented): 23368
Dev entries  (original)  : 185
Test entries (original)  : 34


In [2]:
# ── Reconstruct original dataset ───────────────────────────────────────────────
# Augmented filenames follow: <augtype>_NNNN_SXX_TXX
# The original base ID is always the NNNN_SXX_TXX suffix

originals = {}  # key: base_id -> {name, text, video_path, split}

# Extract originals from train augmented names
for k, v in train_labels.items():
    m = re.search(r'(\d{4}_S\d+_T\d+)$', k)
    if m:
        base_id = m.group(1)
        if base_id not in originals:
            originals[base_id] = {
                'name': base_id,
                'text': v['text'],
                'video_path': base_id + '.mp4',
                'split': 'train'
            }

# Add dev originals
for k, v in dev_labels.items():
    originals[k] = {'name': k, 'text': v['text'], 'video_path': v['video_path'], 'split': 'dev'}

# Add test originals
for k, v in test_labels.items():
    originals[k] = {'name': k, 'text': v['text'], 'video_path': v['video_path'], 'split': 'test'}

print(f"Total original videos    : {len(originals)}")
print(f"  → from train (augmented): {sum(1 for v in originals.values() if v['split']=='train')}")
print(f"  → from dev              : {sum(1 for v in originals.values() if v['split']=='dev')}")
print(f"  → from test             : {sum(1 for v in originals.values() if v['split']=='test')}")
print(f"\nUnique words/classes     : {len(set(v['text'] for v in originals.values()))}")

Total original videos    : 1235
  → from train (augmented): 1016
  → from dev              : 185
  → from test             : 34

Unique words/classes     : 692


In [3]:
# ── Per-word stats ─────────────────────────────────────────────────────────────
# Signer ID is the S-part of the filename: NNNN_S01_T01 → signer = S01

word_stats = defaultdict(lambda: {'videos': [], 'signers': set()})

for vid_id, info in originals.items():
    word = info['text']
    word_stats[word]['videos'].append(info['video_path'])
    
    # Extract signer ID
    m = re.search(r'_(S\d+)_', vid_id)
    if m:
        word_stats[word]['signers'].add(m.group(1))

# Convert to list of dicts for DataFrame
rows = []
for word, stats in word_stats.items():
    rows.append({
        'word':          word,
        'num_videos':    len(stats['videos']),
        'num_signers':   len(stats['signers']),
        'signers':       sorted(stats['signers']),
        'videos':        sorted(stats['videos']),
    })

df = pd.DataFrame(rows).sort_values('num_videos', ascending=False).reset_index(drop=True)

print("Top 10 words by video count:")
print(df[['word','num_videos','num_signers','signers']].head(10).to_string(index=False))

Top 10 words by video count:
  word  num_videos  num_signers              signers
  معلم          12            4 [S01, S02, S05, S07]
   سيئ           9            1                [S05]
  طالب           8            3      [S01, S02, S05]
 مدرسة           7            4 [S01, S02, S03, S07]
  ممكن           7            2           [S05, S09]
  كتاب           7            4 [S01, S02, S03, S06]
   جيد           7            1                [S05]
مساعدة           6            2           [S04, S05]
  دفتر           6            4 [S01, S02, S03, S06]
  سؤال           6            4 [S01, S02, S03, S06]


In [4]:
# ── Coverage summary ───────────────────────────────────────────────────────────
total_videos  = df['num_videos'].sum()
total_classes = len(df)

c1  = df[df['num_videos'] == 1]
c2  = df[df['num_videos'] == 2]
c3p = df[df['num_videos'] >= 3]

s1  = df[df['num_signers'] == 1]
s2p = df[df['num_signers'] >= 2]

# Classes safe to split: need at least 3 videos (1 train, 1 dev, 1 test)
safe_split = df[df['num_videos'] >= 3]
train_only  = df[df['num_videos'] < 3]

print(f"Total original videos : {total_videos}")
print(f"Total classes (words) : {total_classes}")
print()
print("── Video count distribution ──")
print(f"  Classes with 1 video : {len(c1):>4}  ({len(c1)/total_classes*100:.1f}%)")
print(f"  Classes with 2 videos: {len(c2):>4}  ({len(c2)/total_classes*100:.1f}%)")
print(f"  Classes with 3+videos: {len(c3p):>4}  ({len(c3p)/total_classes*100:.1f}%)")
print()
print("── Signer distribution ──")
print(f"  Classes with 1 signer : {len(s1):>4}  ({len(s1)/total_classes*100:.1f}%)")
print(f"  Classes with 2+ signers: {len(s2p):>4}  ({len(s2p)/total_classes*100:.1f}%)")
print()
print("── Splitability ──")
print(f"  Safe for train/dev/test split (>=3 videos): {len(safe_split)}")
print(f"  Train-only (too few videos for split)     : {len(train_only)}")

Total original videos : 1235
Total classes (words) : 692

── Video count distribution ──
  Classes with 1 video :  404  (58.4%)
  Classes with 2 videos:  158  (22.8%)
  Classes with 3+videos:  130  (18.8%)

── Signer distribution ──
  Classes with 1 signer :  549  (79.3%)
  Classes with 2+ signers:  143  (20.7%)

── Splitability ──
  Safe for train/dev/test split (>=3 videos): 130
  Train-only (too few videos for split)     : 562


In [5]:
# ── Bottom 10 (least data) ─────────────────────────────────────────────────────
print("\nBottom 10 words by video count (most at risk):")
print(df[['word','num_videos','num_signers','signers']].tail(10).to_string(index=False))


Bottom 10 words by video count (most at risk):
   word  num_videos  num_signers signers
من فضلك           1            1   [S07]
    يزن           1            1   [S06]
    تعب           1            1   [S07]
   ضابط           1            1   [S07]
   يدمن           1            1   [S06]
  يخترق           1            1   [S06]
   يرمي           1            1   [S07]
 كيمياء           1            1   [S06]
 موسيقى           1            1   [S06]
   شرطي           1            1   [S07]


In [6]:
# ── Save CSV ───────────────────────────────────────────────────────────────────
csv_path = REPORT_DIR / 'word_video_signer_report.csv'
df_csv = df.copy()
df_csv['signers'] = df_csv['signers'].apply(lambda x: ', '.join(x))
df_csv['videos']  = df_csv['videos'].apply(lambda x: ', '.join(x))
df_csv.to_csv(csv_path, index=False, encoding='utf-8-sig')  # utf-8-sig for Arabic in Excel
print(f"Saved CSV  → {csv_path}")

# ── Save JSON ──────────────────────────────────────────────────────────────────
json_path = REPORT_DIR / 'word_video_signer_report.json'
json_data = []
for _, row in df.iterrows():
    json_data.append({
        'word':        row['word'],
        'num_videos':  int(row['num_videos']),
        'num_signers': int(row['num_signers']),
        'signers':     row['signers'],
        'videos':      row['videos'],
    })
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)
print(f"Saved JSON → {json_path}")

Saved CSV  → C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\word_video_signer_report.csv
Saved JSON → C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\word_video_signer_report.json


In [7]:
# ── Save readable text summary ─────────────────────────────────────────────────
txt_path = REPORT_DIR / 'dataset_split_analysis.txt'

lines = [
    "OSL-Words Original Dataset — Split Analysis",
    "=" * 50,
    "",
    f"Total original videos          : {total_videos}",
    f"Total unique words/classes     : {total_classes}",
    "",
    "Video Count Distribution",
    "-" * 30,
    f"  Classes with 1 video  : {len(c1):>4}  ({len(c1)/total_classes*100:.1f}%)",
    f"  Classes with 2 videos : {len(c2):>4}  ({len(c2)/total_classes*100:.1f}%)",
    f"  Classes with 3+ videos: {len(c3p):>4}  ({len(c3p)/total_classes*100:.1f}%)",
    "",
    "Signer Distribution",
    "-" * 30,
    f"  Classes with 1 signer  : {len(s1):>4}  ({len(s1)/total_classes*100:.1f}%)",
    f"  Classes with 2+ signers: {len(s2p):>4}  ({len(s2p)/total_classes*100:.1f}%)",
    "",
    "Splitability Assessment",
    "-" * 30,
    f"  Safe for train/dev/test split (>=3 original videos): {len(safe_split)} classes",
    f"  Should remain train-only (1-2 original videos)     : {len(train_only)} classes",
    "",
    "NOTE: The current split uses only 34 test samples covering 33 words out of",
    f"      {total_classes} total classes. A proper re-split using all {total_videos} original",
    "      videos would provide more reliable evaluation coverage.",
    "",
    "Top 20 words by video count:",
    "-" * 30,
]
for _, row in df.head(20).iterrows():
    lines.append(f"  {row['word']:<30} videos={row['num_videos']}  signers={row['num_signers']} ({', '.join(row['signers'])})")
lines += ["", "Bottom 20 words by video count (least data):", "-" * 30]
for _, row in df.tail(20).iterrows():
    lines.append(f"  {row['word']:<30} videos={row['num_videos']}  signers={row['num_signers']} ({', '.join(row['signers'])})")

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Saved TXT  → {txt_path}")
print()
print('\n'.join(lines[:30]))

Saved TXT  → C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\notebooks\reports\dataset_split_analysis.txt

OSL-Words Original Dataset — Split Analysis

Total original videos          : 1235
Total unique words/classes     : 692

Video Count Distribution
------------------------------
  Classes with 1 video  :  404  (58.4%)
  Classes with 2 videos :  158  (22.8%)
  Classes with 3+ videos:  130  (18.8%)

Signer Distribution
------------------------------
  Classes with 1 signer  :  549  (79.3%)
  Classes with 2+ signers:  143  (20.7%)

Splitability Assessment
------------------------------
  Safe for train/dev/test split (>=3 original videos): 130 classes
  Should remain train-only (1-2 original videos)     : 562 classes

NOTE: The current split uses only 34 test samples covering 33 words out of
      692 total classes. A proper re-split using all 1235 original
      videos would provide more reliable evaluation coverage.

Top 20 words by video count:
------------------------------
  

In [8]:
# ── Full sorted table (display) ────────────────────────────────────────────────
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 40)
df[['word', 'num_videos', 'num_signers', 'signers']].head(50)

,word,num_videos,num_signers,signers
0,معلم,12,4,"[S01, S02, S05, S07]"
1,سيئ,9,1,[S05]
2,طالب,8,3,"[S01, S02, S05]"
3,مدرسة,7,4,"[S01, S02, S03, S07]"
4,ممكن,7,2,"[S05, S09]"
5,كتاب,7,4,"[S01, S02, S03, S06]"
6,جيد,7,1,[S05]
7,مساعدة,6,2,"[S04, S05]"
8,دفتر,6,4,"[S01, S02, S03, S06]"
9,سؤال,6,4,"[S01, S02, S03, S06]"
